In [25]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [26]:
from llama_index.core import SimpleDirectoryReader


In [29]:
base_folder = "./docs"
reader=SimpleDirectoryReader(base_folder, recursive=True, exclude_hidden=True)

In [30]:
reader.input_files

[PosixPath('/Users/ashishsharma/pbl/docs/dream_data (4).json'),
 PosixPath('/Users/ashishsharma/pbl/docs/dreams_interpretations (2).csv')]

In [31]:
docs=reader.load_data()

In [32]:
len(docs)

2

In [33]:
from llama_index.core.node_parser import SentenceSplitter

node_parser = SentenceSplitter(chunk_size=200, chunk_overlap=0)

In [34]:
nodes = node_parser.get_nodes_from_documents(docs,show_progess=True)

In [35]:
len(nodes)

1239

In [36]:
import os
import pandas as pd
import json

df = pd.read_csv("/Users/ashishsharma/Desktop/dreams_interpretations.csv")
dream_data = df.to_dict(orient="records")

#with open("dream_data.json", "w") as f:
    #json.dump(dream_data, f, indent=4)
    #print("File exists:", os.path.exists("dream_data.json"))



In [37]:
import os
from groq import Groq
# with open("./groq_api_key.txt", "r") as f:
groq_api_key = "53c8d2b7-3c0d-460a-9106-1b45b4ed41cf"  
llm = Groq(api_key=groq_api_key)

print("Groq API Key exists:", bool(groq_api_key))


Groq API Key exists: True


In [38]:
from llama_index.core import VectorStoreIndex

In [39]:
!rm -rf ./storage

In [40]:
index_dir = "./storage"

In [41]:
pip install spacy


Note: you may need to restart the kernel to use updated packages.


In [42]:
import spacy
nlp = spacy.load("en_core_web_sm")




In [43]:
import os
import re
import json
import spacy
import pandas as pd
import streamlit as st
from groq import Groq

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Load CSV data
df = pd.read_csv("/Users/ashishsharma/Desktop/dreams_interpretations.csv")
dream_data = df.to_dict(orient="records")

# Initialize Groq API
groq_api_key = "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"  # Make sure this is secure in production
llm = Groq(api_key=groq_api_key)  # Correct API initialization

# Keyword extractor
def extract_keywords(text):
    doc = nlp(text)
    keywords = set()
    for chunk in doc.noun_chunks:
        keywords.add(chunk.text.lower())
    for token in doc:
        if token.pos_ in ("NOUN", "PROPN", "VERB"):
            keywords.add(token.lemma_.lower())
    return list(keywords)

# Dream interpreter
def interpret_dream(user_input):
    keywords = extract_keywords(user_input)
    keyword_set = set(keywords)
    matched_entry = None

    for entry in dream_data:
        symbol = str(entry["Dream Symbol"]).lower()
        if symbol in keyword_set:
            matched_entry = entry
            break

    if matched_entry:
        return matched_entry["Interpretation"]
    else:
        system_instruction = (
            "You are a dream assistant. Always respond directly to the user in plain, active voice. "
            "Do not include internal thoughts, reasoning, or any <think>...</think> tags. "
            "If the dream is not found in the database, politely say so and suggest common dream topics like flying, falling, or losing teeth. "
            "Keep responses short, friendly, and clear."
        )

        chat_completion = llm.chat.completions.create(
            model="deepseek-r1-distill-llama-70b",
            temperature=0.7,
            max_tokens=300,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_input},
            ],
        )

        response = chat_completion.choices[0].message.content
        cleaned_response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
        return cleaned_response

# Streamlit UI
st.title("🌙 Dream Interpreter Bot")
st.write("Describe your dream and I’ll try to interpret it.")

user_input = st.text_input("What did you dream about?")

if user_input:
    response = interpret_dream(user_input)
    st.markdown("**💭 Interpretation:**")
    st.write(response)


2025-04-14 12:57:58.244 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.244 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 12:57:58.246 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar